# Ablación Multi-LoRA sin Verificador (`enrutador + recuperador + redactor`)

Rama aislada: **`exp/multi-lora-sin-verificador`** (deja intactos `exp/multi-lora-todos`, `Deliverable 4/`, `deliverable-4` y `main`).

En el experimento anterior (`exp/multi-lora-todos`), entrenar los 4 agentes subió el rendimiento en casos difíciles (`18/25 = 72%` en `dificil_25` y `14/20 = 70%` en `nuevas_20`) pero el **Verificador LoRA** rechazó 17 respuestas válidas en `75 D3` provocando regresión en `congelado_50` (`40/50`) y mayor latencia.

Aquí reutilizamos exactamente los adaptadores ya entrenados de **`enrutador`** y **`recuperador`** junto al **`redactor` D4 v2** (`Pablox0333/qwen3-4b-redactor-udec-d4`, revisión `739b031acbd5feef2c49f946b94722712209bce0`), pero dejamos al **`verificador` en modo base (`Qwen/Qwen3-4B`, sin LoRA)**.

## 1. Entorno, GPU, repositorio (`exp/multi-lora-sin-verificador`) y dependencias fijadas

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1.git"
REPO_BRANCH = "exp/multi-lora-sin-verificador"
REPO_DIR = Path("/content/Proyecto_Git")

if not REPO_DIR.exists():
    subprocess.check_call([
        "git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)
    ])
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=False)

os.chdir(REPO_DIR)
print("Branch actual:", subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip())
print("Commit actual:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

subprocess.check_call(["nvidia-smi"])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==5.23.0",
    "peft==0.21.3",
    "bitsandbytes==0.50.2",
    "accelerate==1.15.0",
    "huggingface-hub==1.33.0",
    "sentence-transformers==6.3.0",
    "matplotlib",
])
print("Dependencias instaladas correctamente.")

## 2. Restaurar adaptadores `enrutador` y `recuperador` desde Google Drive (sin reentrenar salvo si faltan)

In [ ]:
import gc
import os
import shutil
import subprocess
import sys
from pathlib import Path
import torch

REPO_DIR = Path("/content/Proyecto_Git")
os.chdir(REPO_DIR)

USAR_GOOGLE_DRIVE = True
DRIVE_DIR = Path("/content/drive/MyDrive/UdeC_GenIA_Exp_MultiLoRA")
REENTRENAR_SI_FALTA = True
ROLES_ENTRENAMIENTO = ("enrutador", "recuperador")

if USAR_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        for ag in ROLES_ENTRENAMIENTO:
            src_ad = DRIVE_DIR / "entrenamiento" / ag / "adapter"
            dst_ad = REPO_DIR / "exp_multi_lora" / "entrenamiento" / ag / "adapter"
            if (src_ad / "adapter_model.safetensors").is_file():
                dst_ad.mkdir(parents=True, exist_ok=True)
                shutil.copytree(src_ad, dst_ad, dirs_exist_ok=True)
                print(f"[{ag}] Restaurado desde Drive: {src_ad}")
        src_eval = DRIVE_DIR / "evaluacion"
        dst_eval = REPO_DIR / "exp_multi_lora" / "evaluacion"
        dst_eval.mkdir(parents=True, exist_ok=True)
        if src_eval.exists():
            for p in src_eval.glob("*sin_verificador*"):
                if p.is_file():
                    shutil.copy2(p, dst_eval / p.name)
                    print(f"Reanudación restaurada desde Drive: {p.name}")
    except Exception as exc:
        print("Drive no disponible:", exc)

for ag in ROLES_ENTRENAMIENTO:
    weights = REPO_DIR / "exp_multi_lora" / "entrenamiento" / ag / "adapter" / "adapter_model.safetensors"
    if weights.is_file():
        print(f"[{ag}] Pesos listos ({weights.stat().st_size / (1024**2):.1f} MB): {weights}")
        continue
    if not REENTRENAR_SI_FALTA:
        raise FileNotFoundError(f"Faltan pesos de {ag} en {weights}")
    print(f"[{ag}] No se encontraron pesos en local/Drive; entrenando {ag}...")
    subprocess.check_call([
        sys.executable, "exp_multi_lora/scripts/entrenar_agente.py", "--agente", ag, "--skip-smoke-test"
    ])
    if USAR_GOOGLE_DRIVE and DRIVE_DIR.exists():
        src_ad = REPO_DIR / "exp_multi_lora" / "entrenamiento" / ag / "adapter"
        dst_ad = DRIVE_DIR / "entrenamiento" / ag / "adapter"
        dst_ad.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src_ad, dst_ad, dirs_exist_ok=True)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def _respaldar_sin_verificador_a_drive() -> None:
    if not (USAR_GOOGLE_DRIVE and DRIVE_DIR.exists()):
        return
    dst_eval = DRIVE_DIR / "evaluacion"
    dst_eval.mkdir(parents=True, exist_ok=True)
    for p in (REPO_DIR / "exp_multi_lora" / "evaluacion").glob("*sin_verificador*"):
        if p.is_file():
            shutil.copy2(p, dst_eval / p.name)

## 3. Carga Multi-LoRA sin Verificador (`enrutador`, `recuperador`, `redactor`) y prueba de aislamiento

In [ ]:
import json
import os
import sys
from pathlib import Path
import torch

os.chdir("/content/Proyecto_Git")
for p_str in (
    str(Path("Deliverable 4/scripts").resolve()),
    str(Path("Deliverable 3/Deliverable3/agentes").resolve()),
    str(Path("exp_multi_lora/scripts").resolve()),
):
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

import evaluar_d4 as ed4
import evaluar_multi_lora as eml
import resumir_comparativa as rc

ROLES_ACTIVOS = ("enrutador", "recuperador", "redactor")
ADAPTADORES_MAP = {"enrutador": "enrutador", "recuperador": "recuperador", "redactor": "redactor", "verificador": None}
fuentes = eml.resolver_fuentes_adaptadores_default()

corpus = ed4.construir_corpus_d3(use_e5=True)
model, tokenizer, compute_dtype_str, base_commit = eml.cargar_modelo_y_multi_lora(
    eml.MODELO_BASE_DEFAULT,
    fuentes,
    ROLES_ACTIVOS,
)
aislamiento = eml.verificar_aislamiento_multi_lora(model, tokenizer, ROLES_ACTIVOS)
print("Compute dtype:", compute_dtype_str)
print("Roles con LoRA activo:", ROLES_ACTIVOS, "| verificador: BASE (sin LoRA)")
print(json.dumps(aislamiento, ensure_ascii=False, indent=2))

GREEDY = False
EVAL_OUT_DIR = Path("exp_multi_lora/evaluacion")
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
prompts_sha = ed4.verificar_prompts_d4()
meta_sin_ver = {
    "modelo_base": eml.MODELO_BASE_DEFAULT,
    "revision_modelo_base": base_commit,
    "roles_con_lora": list(ROLES_ACTIVOS),
    "roles_en_base": ["verificador"],
    "fuentes_por_rol": {r: {"fuente": fuentes[r][0], "revision": fuentes[r][1]} for r in ROLES_ACTIVOS},
    "hardware": {
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "compute_dtype": compute_dtype_str,
    },
    "greedy": GREEDY,
    "aislamiento_multi_lora": aislamiento,
    "prompts_sha256": prompts_sha,
    "versiones": ed4._resolver_versiones_eval(),
}

## 4. Evaluación 1: `nuevas_20` (20 preguntas inéditas)

In [ ]:
preg_n20 = ed4.cargar_preguntas_desde_csv(
    Path("Deliverable 4/datos/preguntas_nuevas_20.csv"),
    nombre_set="nuevas_20",
)
gen_n20 = eml.MultiLoraGenerator(
    model, tokenizer, adaptadores_activos=ADAPTADORES_MAP, greedy=GREEDY
)
ed4.ejecutar_evaluacion_lote(
    preg_n20,
    corpus,
    gen_n20,
    output_dir=EVAL_OUT_DIR,
    condicion="multi_lora:enrutador+recuperador+redactor",
    metadata_extra=meta_sin_ver,
    output_prefix="sin_verificador_nuevas20",
)
_respaldar_sin_verificador_a_drive()

res_n20 = [
    rc.resumir_corrida(
        "Control Base D4",
        Path("Deliverable 4/evaluacion/resultados_nuevas20_base_D4_auditados.csv"),
        Path("Deliverable 4/evaluacion/telemetria_nuevas20_base_D4.json"),
    ),
    rc.resumir_corrida(
        "Solo Redactor D4",
        Path("Deliverable 4/evaluacion/resultados_nuevas20_ft_D4_auditados.csv"),
        Path("Deliverable 4/evaluacion/telemetria_nuevas20_ft_D4.json"),
    ),
    rc.resumir_corrida(
        "Multi-LoRA (4 agentes)",
        EVAL_OUT_DIR / "resultados_multi_lora_nuevas20_auditados.csv",
        EVAL_OUT_DIR / "telemetria_multi_lora_nuevas20.json",
    ),
    rc.resumir_corrida(
        "3-LoRA (Sin Verificador)",
        EVAL_OUT_DIR / "resultados_sin_verificador_nuevas20.csv",
        EVAL_OUT_DIR / "telemetria_sin_verificador_nuevas20.json",
    ),
]
print("### Comparativa nuevas_20 ###")
print(rc.formatear_tabla_markdown(res_n20))
print("Desglose adaptadores (3-LoRA):", json.dumps(res_n20[-1]["telemetria"]["llamadas_por_adaptador"], indent=2))

## 5. Evaluación 2: Benchmark completo `75` D3 (`congelado_50` + `dificil_25`) y paquete descargable

In [ ]:
import zipfile

preg_75 = ed4.cargar_preguntas_evaluacion()
gen_75 = eml.MultiLoraGenerator(
    model, tokenizer, adaptadores_activos=ADAPTADORES_MAP, greedy=GREEDY
)
ed4.ejecutar_evaluacion_lote(
    preg_75,
    corpus,
    gen_75,
    output_dir=EVAL_OUT_DIR,
    condicion="multi_lora:enrutador+recuperador+redactor",
    metadata_extra=meta_sin_ver,
    output_prefix="sin_verificador_75",
)
_respaldar_sin_verificador_a_drive()

res_75 = [
    rc.resumir_corrida(
        "Control Base D4",
        Path("Deliverable 4/evaluacion/resultados_control_base_D4_auditados.csv"),
        Path("Deliverable 4/evaluacion/telemetria_control_base_D4.json"),
    ),
    rc.resumir_corrida(
        "Solo Redactor D4",
        Path("Deliverable 4/evaluacion/resultados_agentes_D4_auditados.csv"),
        Path("Deliverable 4/evaluacion/telemetria_llamadas.json"),
    ),
    rc.resumir_corrida(
        "Multi-LoRA (4 agentes)",
        EVAL_OUT_DIR / "resultados_multi_lora_75_auditados.csv",
        EVAL_OUT_DIR / "telemetria_multi_lora_75.json",
    ),
    rc.resumir_corrida(
        "3-LoRA (Sin Verificador)",
        EVAL_OUT_DIR / "resultados_sin_verificador_75.csv",
        EVAL_OUT_DIR / "telemetria_sin_verificador_75.json",
    ),
]
print("### Comparativa 75 preguntas D3 ###")
print(rc.formatear_tabla_markdown(res_75))
print("Desglose adaptadores (75 D3):", json.dumps(res_75[-1]["telemetria"]["llamadas_por_adaptador"], indent=2))

zip_path = Path("/content/paquete_sin_verificador.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(EVAL_OUT_DIR.glob("*sin_verificador*")):
        if p.is_file():
            zf.write(p, arcname=p.name)
print(f"ZIP listo: {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")
if USAR_GOOGLE_DRIVE and DRIVE_DIR.exists():
    shutil.copy2(zip_path, DRIVE_DIR / "paquete_sin_verificador.zip")
    print("Copiado también a Drive:", DRIVE_DIR / "paquete_sin_verificador.zip")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass